# 01 · Data Cleaning & Preparation

**Project:** Supply Chain & Logistics Performance Analytics

**Goal of this notebook:** load the raw order-item data, check its quality, clean it, and
create the derived logistics features (delay days, on-time flag, time columns) used by the
rest of the analysis.

> **Data source.** The project follows the schema of the public
> [DataCo Smart Supply Chain dataset](https://www.kaggle.com/datasets/shashwatwork/dataco-smart-supply-chain-for-big-data-analysis).
> The repo ships with a synthetic sample in the same format so everything runs out of the box.
> Put the real file at `data/raw/DataCoSupplyChainDataset.csv` and re-run - `load_raw()` picks it up automatically.

In [1]:
import sys
sys.path.append("../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

from config import CLEAN_ITEMS_PATH, CLEAN_ORDERS_PATH, IMAGES
from viz import set_style, label_bars, BLUE, ORANGE, AQUA, MUTED, TEXT_2

set_style()
pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", 40)

from cleaning import load_raw, clean, to_orders, COLUMN_MAP

## 1. Load raw data

In [2]:
raw, source = load_raw()
print(f"Source : {source}")
print(f"Shape  : {raw.shape[0]:,} rows x {raw.shape[1]} columns")
raw.head()

Source : synthetic sample (DataCo schema)
Shape  : 24,776 rows x 29 columns

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Name,Customer Segment,Department Name,Market,Order City,Order Country,order date (DateOrders),Order Id,Order Item Id,Order Item Discount,Order Item Discount Rate,Order Item Product Price,Order Item Quantity,Sales,Order Item Total,Order Profit Per Order,Order Region,Order Status,Product Name,Product Price,shipping date (DateOrders),Shipping Mode
0,PAYMENT,4,4,51.22,297.49,Shipping on time,0,Electronics,Home Office,Outdoors,USCA,United States,United States,4/28/2017 6:07,1,1,52.50,0.15,349.99,1,349.99,297.49,51.22,US Center,COMPLETE,Garmin Forerunner 910XT GPS Watch,349.99,5/2/2017 6:07,Standard Class
1,PAYMENT,4,4,33.23,127.99,Shipping on time,0,Soccer,Home Office,Fan Shop,USCA,United States,United States,4/28/2017 6:07,1,2,32.00,0.20,159.99,1,159.99,127.99,33.23,US Center,COMPLETE,adidas Brazuca 2014 Official Match Ball,159.99,5/2/2017 6:07,Standard Class
2,PAYMENT,4,4,37.50,199.99,Shipping on time,0,Water Sports,Home Office,Fan Shop,USCA,United States,United States,4/28/2017 6:07,1,3,0.00,0.00,199.99,1,199.99,199.99,37.50,US Center,COMPLETE,Pelican Sunstream 100 Kayak,199.99,5/2/2017 6:07,Standard Class
3,PAYMENT,4,4,49.17,127.50,Shipping on time,0,Women's Apparel,Home Office,Golf,USCA,United States,United States,4/28/2017 6:07,1,4,22.50,0.15,50.00,3,150.00,127.50,49.17,US Center,COMPLETE,Nike Men's Dri-FIT Victory Golf Polo,50.00,5/2/2017 6:07,Standard Class
4,PAYMENT,3,4,-3.55,45.50,Advance shipping,0,Women's Apparel,Consumer,Golf,LATAM,Colombia,Colombia,9/5/2017 20:38,2,5,4.50,0.09,50.00,1,50.00,45.50,-3.55,South America,ON_HOLD,Nike Men's Dri-FIT Victory Golf Polo,50.00,9/8/2017 20:38,Standard Class


In [3]:
raw[list(COLUMN_MAP)].dtypes.to_frame("dtype").T

,Order Id,Order Item Id,order date (DateOrders),shipping date (DateOrders),Type,Days for shipping (real),Days for shipment (scheduled),Delivery Status,Late_delivery_risk,Shipping Mode,Market,Order Region,Order Country,Customer Segment,Department Name,Category Name,Product Name,Product Price,Order Item Quantity,Order Item Discount,Order Item Discount Rate,Sales,Order Item Total,Order Profit Per Order,Order Status
dtype,int64,int64,str,str,str,int64,int64,str,int64,str,str,str,str,str,str,str,str,float64,int64,float64,float64,float64,float64,float64,str


## 2. Data quality checks

In [4]:
quality = pd.DataFrame({
    "missing_values": raw.isna().sum(),
    "missing_%": 100 * raw.isna().mean(),
    "unique_values": raw.nunique(),
})
quality[quality["missing_values"] > 0]

,missing_values,missing_%,unique_values
Customer Segment,248,1.00,3


In [5]:
print("Fully duplicated rows      :", raw.duplicated().sum())
print("Duplicated order item IDs  :", raw["Order Item Id"].duplicated().sum())
print("Negative shipping days     :", (raw["Days for shipping (real)"] < 0).sum())
print("Negative sales             :", (raw["Sales"] < 0).sum())
print("Order lines with a loss (profit<0):", (raw["Order Profit Per Order"] < 0).sum())

Fully duplicated rows      : 122
Duplicated order item IDs  : 123
Negative shipping days     : 0
Negative sales             : 0
Order lines with a loss (profit<0): 8099

**Findings**
- A small number of fully duplicated rows (same order item repeated) - these would double-count sales, so they are dropped.
- `Customer Segment` has ~1% missing values - filled with `"Unknown"` so no orders are lost.
- No impossible values (negative days or sales). Negative profit is valid (loss-making orders) and is kept.
- Dates are stored as text (`m/d/Y H:M`) and need to be parsed.

## 3. Clean and engineer features

In [6]:
items = clean(raw)
orders = to_orders(items)

print(f"Item rows after cleaning : {len(items):,}  (removed {len(raw) - len(items):,})")
print(f"Unique orders            : {len(orders):,}")
print(f"Date range               : {items['order_date'].min():%d %b %Y} -> {items['order_date'].max():%d %b %Y}")
items.head()

Item rows after cleaning : 24,653  (removed 123)
Unique orders            : 12,000
Date range               : 01 Jan 2015 -> 31 Dec 2017

,order_id,order_item_id,order_date,shipping_date,payment_type,actual_days,scheduled_days,delivery_status,is_late,shipping_mode,market,region,country,customer_segment,department,category,product,product_price,quantity,discount,discount_rate,gross_sales,net_sales,profit,order_status,delay_days,is_cancelled,is_on_time,is_loss,profit_margin,order_year,order_month,order_quarter,order_weekday,order_hour
0,1,1,2017-04-28 06:07:00,2017-05-02 06:07:00,PAYMENT,4,4,Shipping on time,0,Standard Class,USCA,US Center,United States,Home Office,Outdoors,Electronics,Garmin Forerunner 910XT GPS Watch,349.99,1,52.50,0.15,349.99,297.49,51.22,COMPLETE,0,0,1,0,0.17,2017,2017-04,2017Q2,Friday,6
1,1,2,2017-04-28 06:07:00,2017-05-02 06:07:00,PAYMENT,4,4,Shipping on time,0,Standard Class,USCA,US Center,United States,Home Office,Fan Shop,Soccer,adidas Brazuca 2014 Official Match Ball,159.99,1,32.00,0.20,159.99,127.99,33.23,COMPLETE,0,0,1,0,0.26,2017,2017-04,2017Q2,Friday,6
2,1,3,2017-04-28 06:07:00,2017-05-02 06:07:00,PAYMENT,4,4,Shipping on time,0,Standard Class,USCA,US Center,United States,Home Office,Fan Shop,Water Sports,Pelican Sunstream 100 Kayak,199.99,1,0.00,0.00,199.99,199.99,37.50,COMPLETE,0,0,1,0,0.19,2017,2017-04,2017Q2,Friday,6
3,1,4,2017-04-28 06:07:00,2017-05-02 06:07:00,PAYMENT,4,4,Shipping on time,0,Standard Class,USCA,US Center,United States,Home Office,Golf,Women's Apparel,Nike Men's Dri-FIT Victory Golf Polo,50.00,3,22.50,0.15,150.00,127.50,49.17,COMPLETE,0,0,1,0,0.39,2017,2017-04,2017Q2,Friday,6
4,2,5,2017-09-05 20:38:00,2017-09-08 20:38:00,PAYMENT,3,4,Advance shipping,0,Standard Class,LATAM,South America,Colombia,Consumer,Golf,Women's Apparel,Nike Men's Dri-FIT Victory Golf Polo,50.00,1,4.50,0.09,50.00,45.50,-3.55,ON_HOLD,-1,0,1,1,-0.08,2017,2017-09,2017Q3,Tuesday,20


**Derived columns**

| Column | Meaning |
|---|---|
| `delay_days` | actual shipping days − scheduled days (positive = late) |
| `is_late` | 1 if delivered after the promised date |
| `is_on_time` | 1 if delivered on or before the promised date (and not cancelled) |
| `is_cancelled` | 1 if order was cancelled or flagged as suspected fraud |
| `profit_margin` | profit ÷ net sales |
| `order_month / quarter / weekday / hour` | time features for trend analysis |

In [7]:
items[["actual_days", "scheduled_days", "delay_days", "quantity", "net_sales", "profit", "discount_rate"]].describe().T

,count,mean,std,min,25%,50%,75%,max
actual_days,"24,653.00",3.56,1.63,0.00,2.00,3.00,5.00,8.00
scheduled_days,"24,653.00",2.94,1.37,0.00,2.00,4.00,4.00,4.00
delay_days,"24,653.00",0.62,1.48,-2.00,0.00,1.00,2.00,5.00
quantity,"24,653.00",1.67,1.19,1.00,1.00,1.00,2.00,5.00
net_sales,"24,653.00",158.91,174.48,7.49,48.98,107.89,199.17,"1,500.00"
profit,"24,653.00",17.56,60.44,-722.81,-4.21,8.58,31.37,750.00
discount_rate,"24,653.00",0.10,0.07,0.00,0.04,0.10,0.17,0.25


## 4. Save processed data

In [8]:
CLEAN_ITEMS_PATH.parent.mkdir(parents=True, exist_ok=True)
items.to_csv(CLEAN_ITEMS_PATH, index=False)
orders.to_csv(CLEAN_ORDERS_PATH, index=False)
print("Saved:", CLEAN_ITEMS_PATH.name, "and", CLEAN_ORDERS_PATH.name)

Saved: order_items_clean.csv and orders_clean.csv